# Step 1 benchmark on a Colab T4Same benchmark as `bench/run_demo.sh` on the Mac, run on CUDA so the step-1 report's"Colab T4" column can be filled in. Nothing here is Mac-specific: `memlayers.devices`picks `cuda > mps > cpu`, so the code path is identical apart from the device.**Before running:** Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU**.The notebook does four things:1. install the pinned requirements2. get the repo (git clone, or upload a zip)3. run probe / speed / read-log / learning phases4. print the summary table and offer the result JSONs for download

## 1. Check the GPU

In [ ]:
!nvidia-smiimport torchprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())if torch.cuda.is_available():    p = torch.cuda.get_device_properties(0)    print(f"{p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")    assert "T4" in p.name or True, "not a T4 -- numbers will not be comparable"else:    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU.")

## 2. Get the repoEither clone it (set `REPO_URL` to the team's GitHub remote) or upload a zip of the`final_project` directory. The zip path is the fallback while the repo has no remote.

In [ ]:
import os, pathlib, subprocessREPO_URL = ""          # e.g. "https://github.com/<team>/<repo>.git"PROJECT  = "final_project"if REPO_URL:    if not pathlib.Path(PROJECT).exists():        subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)else:    # upload a zip containing memlayers/, bench/, tools/, tests/, requirements.txt    from google.colab import files    up = files.upload()    name = next(iter(up))    subprocess.run(["unzip", "-q", "-o", name], check=True)    # the zip may or may not contain the top directory    if not pathlib.Path(PROJECT).exists():        PROJECT = "."os.chdir(PROJECT)print("cwd:", os.getcwd())print(sorted(p.name for p in pathlib.Path('.').iterdir())[:20])

## 3. InstallColab already ships a CUDA torch, so torch is deliberately **not** reinstalled -- thatwould cost several minutes and risk a CUDA mismatch. Only the small packages the codeactually imports are installed.

In [ ]:
!pip -q install product-key-memory PEER-pytorch pyyaml einopsimport torch, yaml, einopsfrom product_key_memory import PKMfrom PEER_pytorch import PEERprint("imports OK | torch", torch.__version__)

In [ ]:
# sanity: the model builds and a step runs on cudaimport sys; sys.path.insert(0, ".")from memlayers.train import Trainer, load_configfrom memlayers.devices import pick_deviceprint("device:", pick_device())tr = Trainer(load_config("memlayers/configs/t1_pkm.yaml",                         ["train.steps=3", "train.micro_batch_size=4",                          "train.log_every=0", "runtime.out_dir=runs/_colab_smoke"]))print("params:", tr.model.param_counts())for _ in range(3):    row = tr.train_step()print(f"ok: {row['loss']:.3f} loss, {row['step_time_s']*1000:.0f} ms/step")

## 4. Run the benchmarkPhases match the Mac run. The T4 has 16 GB, so the micro-batch probe may well reach 32where the Mac stops earlier -- that is one of the numbers we want.`run_demo.sh` works here too, but calling the phases from Python keeps the output in thenotebook and lets us drop the big configs if the session is short.

In [ ]:
import json, timefrom bench.benchmark import probe, speed, learn, summarize, env_infoprint(json.dumps(env_info(), indent=2))ALL     = ["t1_dense", "t1_pkm", "t1_peer", "t1_pkm_frozen",           "t2_dense", "t2_pkm", "t2_pkm_262k", "t2_peer"]MEMORY  = ["t1_pkm", "t1_peer", "t1_pkm_frozen", "t2_pkm", "t2_pkm_262k", "t2_peer"]LEARN   = ["t1_pkm", "t1_peer"]STEPS, WARMUP, LEARN_STEPS = 300, 30, 1000cfg = lambda n: f"memlayers/configs/{n}.yaml"

In [ ]:
# PHASE 1 -- micro-batch probemicro = {}for n in ALL:    r = probe(cfg(n))    micro[n] = r["largest_micro_batch"]    print(f"{n:16s} largest micro-batch: {micro[n]}")

In [ ]:
# PHASE 2 -- speed and memory on fake data (a)for n in ALL:    t0 = time.time()    r = speed(cfg(n), steps=STEPS, warmup=WARMUP, micro=micro[n])    print(f"{n:16s} {r['ms_per_step_mean']:7.1f} ms/step  "          f"{r['tokens_per_s_from_mean_time']:8.0f} tok/s  "          f"{r['peak_mem_GB']:5.2f} GB  ({time.time()-t0:.0f}s)")

In [ ]:
# PHASE 3 -- the same with read logging onfor n in MEMORY:    r = speed(cfg(n), steps=STEPS, warmup=WARMUP, micro=micro[n], read_logging=True)    rl = r.get("readlog", {})    print(f"{n:16s} {r['ms_per_step_mean']:7.1f} ms/step  "          f"readlog {rl.get('GB_per_million_tokens', float('nan')):.3f} GB/Mtok")

In [ ]:
# PHASE 4 -- learning check on fake data (b)for n in LEARN:    r = learn(cfg(n), steps=LEARN_STEPS, micro=micro[n])    print(f"{n:16s} loss {r['loss_first']:.3f} -> {r['loss_final']:.3f}  "          f"(floor {r['task_loss_floor_nats']:.3f}, random {r['random_token_level_nats']:.3f})  "          f"below random: {r['clearly_below_random']}")    for layer, st in (r.get("slot_usage") or {}).items():        print(f"    layer {layer}: {st['fraction_used']*100:.1f}% of slots read, "              f"KL(usage || uniform) = {st['kl_from_uniform']:.3f} nats "              f"(max {st['max_kl_nats']:.3f})")

## 5. Summary table and extrapolation

In [ ]:
out = summarize()

In [ ]:
# hours per run and per 12-run tier, using the T4 numbers just measuredTIER_RUNS = 12for r in out["rows"]:    if r["read_logging"]:        continue    tok = 150e6 if r["name"].startswith("t1") else 500e6    h = r["hours_150M"] if r["name"].startswith("t1") else r["hours_500M"]    print(f"{r['name']:16s} {tok/1e6:5.0f}M tokens  {h:6.2f} h/run  "          f"{h*TIER_RUNS:7.1f} h for {TIER_RUNS} runs")

## 6. Download the results, to paste into the step-1 report

In [ ]:
import shutilfrom google.colab import filesshutil.make_archive("colab_results", "zip", "bench/results")files.download("colab_results.zip")